Step 1: Bootstrap & install

In [ ]:
!pip install --quiet anthropic pydantic
!rm -rf /content/astra-swarm 2>/dev/null
!git clone --depth 1 -q https://github.com/phdeore/astra-swarm.git /content/astra-swarm

import sys, os
sys.path.insert(0, "/content/astra-swarm/src")

from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()

from typing import Any
result: dict[str, Any] = {}

from astra_swarm import attack_kb
from astra_swarm.tools import lookup_attack_technique_by_id, search_attack_techniques
from astra_swarm.agent_loop import run_with_tools
from astra_swarm.cassette import cassette
print("ready")

Step 2: ID lookup from KB

In [ ]:
# Direct ID lookup — triggers the download on first call
t = lookup_attack_technique_by_id("T1078")
print(t["name"], "→ tactics:", t["tactics"])
print(t["description"][:200], "...")           # Update to get 
print()

# Sub-technique
t = lookup_attack_technique_by_id("T1078.004")
print(t["name"], "→ tactics:", t["tactics"])
print()

# Unknown ID
t = lookup_attack_technique_by_id("T9999")
print(t)
print()

# Keyword search
r = search_attack_techniques(keyword="mfa fatigue")
for m in r.get("matches", []):
    print(f"  {m['id']:<10} {m['name']}")

Step 3: Verify tool round trip

In [ ]:
with cassette("04_tool_round_trip", mode="auto"):
    result = run_with_tools(
        "Explain in one paragraph what would motivate an adversary to combine "
        "MITRE ATT&CK T1078 (Valid Accounts) and T1621 (MFA request generation). "
        "Use the lookup tool to ground each technique. Cite tactics explicitly.",
        verbose=True,
        max_rounds=6,
    )
print()
print(result["final_text"])
print(f"\nrounds: {result['rounds']}, stop_reason: {result['stop_reason']}")

In [ ]:
# with a keyword call

with cassette("04_keyword_call", mode="auto"):
    result = run_with_tools(
        "A user account received 47 push notifications in 5 minutes. "
        "Look up the relevant MITRE technique and explain in one sentence what's happening.",
        verbose=True,
        max_rounds=8,
    )
print(result["final_text"])

Step 4: Rerun full triage chain with enrichment

In [ ]:
import json
from pathlib import Path
from astra_swarm.alerts import triage_chain

try:
    alerts = json.loads(Path("/content/astra-swarm/data/synthetic/02_alerts.json").read_text())
    results = []
    for i, a in enumerate(alerts, 1):
        print(f"--- Alert {i} ---")
        r = triage_chain(a)
        results.append(r)
        techs = r.attack.techniques
        print(f"  Cited ATT&CK: {', '.join(t.id for t in techs) or '(none)'}")
        print(f"  Severity: {r.verdict.severity} "
            f"(conf {r.verdict.confidence:.2f})")
        print()
except FileNotFoundError:
    print("Alert file not found.")